In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import os
import sys
import json
import random

import numpy as np 
import pandas as pd 

import librosa as lb
import librosa.feature as lf
import librosa.display as ld
import soundfile as sf
import kagglehub 


import matplotlib.pyplot as plt
from IPython.display import Audio
from tqdm import tqdm

import torch
import torchaudio

# Kaggle Set-up
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
#os.environ['HF_TOKEN'] = user_secrets.get_secret("hf_access")
os.environ["KAGGLE_USERNAME"] = user_secrets.get_secret("kgg_user")
os.environ["KAGGLE_KEY"] = user_secrets.get_secret("kgg_key")

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

#for dirname, _, filenames in os.walk('/kaggle/input'):
    #for filename in filenames:
        #print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# CONFIGURATION
DATA_ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems'
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop','jazz', 'metal', 'pop', 'reggae', 'rock'] 
STEMS = {'vocals.wav','other.wav','bass.wav','drums.wav'} 
STEM_KEYS = ['drums', 'vocals', 'bass', 'other']
SONG_INDEX = ''  

# NOISE DATASET
root_dir = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/'
csv_file = '/meta/esc50.csv'
audio_folder = '/audio/'


SR = 22050
DURATION = 30

#============ Check for GPU availability ==============================#
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Using device: {device}")

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
if torch.cuda.is_available():
    print("GPU is available.Setting RANDOM_SEED .... ")
        # setting for both CPU and GPU
    torch.cuda.manual_seed_all(RANDOM_SEED)
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("   Running on CPU")
print("\n✅ Environment setup complete!")

import warnings
warnings.filterwarnings("ignore")


🚀 Using device: cpu
   Running on CPU

✅ Environment setup complete!


# Classical ML Baseline

* Clean and preprocess audio stems.
* Convert audio to numerical features (MFCCs, Spectrograms).
* Train Logistic Regression, Naive Bayes, or boosting models (CatBoost, LightGBM, XGBoost).
* Evaluate using Macro F1 and log results in W&B.

# Defined functions

In [2]:
def load_and_fix(path,sr=SR,duration=DURATION):
    LENGTH = sr*duration    
    waveform, _sr_ = torchaudio.load(path)

    # Convert to mono
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    # Resample if needed
    if _sr_ != sr:
        resampler = torchaudio.transforms.Resample(_sr_, sr)
        waveform = resampler(waveform)

    waveform = waveform.squeeze(0)

    # Trim or pad
    if waveform.shape[0] >= LENGTH:
        return waveform[:LENGTH]
    else:
        padding = LENGTH - waveform.shape[0]
        return torch.nn.functional.pad(waveform, (0, padding))
            
def build_dataset(root_dir,seed=42):
    """
        This takes root dirstory and load paths of all stem files as a dictionary.
        Returns a Dictionary.
    """
    global SR
    global DURATION
    LENGTH = SR*DURATION
    
    # Initialize empty dictionaries
    loaded_song = {g: {s.replace('.wav', ''): [] for s in STEMS} for g in GENRES}
    total_stem_file = 0
    for g in tqdm(GENRES,desc="Loading stems into dict.."):
        folder_path = os.path.join(root_dir,g)
        if os.path.exists(folder_path) and os.path.isdir(folder_path):
            for i in range(0,100):
                song_folder = g + '.' + f"{i:05d}"
                for s in STEMS:
                    stem_file_path = os.path.join(folder_path ,song_folder ,s)
                    if os.path.exists(stem_file_path):
                        total_stem_file += 1

                        y = load_and_fix(stem_file_path)
                        loaded_song[g][s.replace('.wav', '')].append(y)
                    else:
                        print(f"Stem file {s} not exists !")
            
        else:
            print(f"Folder '{g}' not exists !")

    print("✅ Total number of stems music file loaded successfully : ", total_stem_file)
    
    return loaded_song


def load_noise_audios(root_dir,audio_folder,csv_file):
    """
     It will extract all noise audio waveform and store in a dictionary.
    """
    noise = {}
    df_noise = pd.read_csv(root_dir+csv_file)
    
    for i in tqdm(df_noise.index,desc="Loading Noise audio ... "):
        filename = df_noise.loc[i]['filename']
        path = root_dir+audio_folder+filename
        y = load_and_fix(path,sr=SR,duration=5)    
        noise[filename] = y
    return noise

def create_single_track(y1,y2,y3,y4):
    """
        Combine 4 stems into a single normalized track.
        All tracks are resampled, padded or trimmed to 30 seconds.
        GPU-compatible.
    """

    mix = y1+y2+y3+y4
    max_amplitude_val = torch.max(torch.abs(mix))
    
    if(max_amplitude_val > 0):
        track = mix/max_amplitude_val
    else:
        return mix
    return track


def stem_recombination(sl,genre):  
    """
        This function randomly pick different stems from same genre and makes a mashup.
        RETURN : single music track (Combination of stems files of different songs from same genre)
    """
    stems = []
    for s in STEM_KEYS:
        stem = random.choice(sl[genre][s])
        stems.append(stem)
    mashup = create_single_track(*stems)
    return mashup
    
def add_noise(mashup,noises):
    """
        It will take recombination music and add noise at random places.
    """
    num_insertions = random.choice([4,5]) 
    noise_audios = random.choices(noises,k=num_insertions)
    
    max_start = 22050*(30-5)  # as sample rate for all is 22050 and noise duration is 5sec and audio duration is 30sec.
    
    positions = random.sample(range(0, max_start), num_insertions)
    #print(num_insertions, noise_files, max_start,np.array(positions)/22050)

    output = torch.clone(mashup)
    for i,pos in enumerate(positions):
        output[pos:pos+len(noise_audios[i])] += noise_audios[i]
    output = output / torch.max(torch.abs(output))
    return output

# Data augmentation function
def data_augmentation(sl,noises,g=None,sample_count=1000):
    """
        Creating actual noisy mashup.
    """
    if g is None:
        return 
    mashup = []
    total_size = 0 #Mb

    print(g)
    # Directory path
    dir_path = f"/kaggle/working/{g}"
    
    # Create directory if it doesn't exist
    try:
        os.makedirs(dir_path, exist_ok=True)
        print(f"Directory created at: {dir_path}")
    except Exception as e:
        print(f"Error creating directory: {e}")
    
    for i in tqdm(range(0,sample_count),desc="Creating noisy mashup"):
        mashup = stem_recombination(sl,g)
        noisy_mashup = add_noise(mashup,noises)
        torchaudio.save(f"{dir_path}/mashup_{i}.wav", noisy_mashup, SR)
        total_size += os.path.getsize(f"{dir_path}/mashup_{i}.wav")/(1024*1024*1024)
    print(f"🤺 {total_size} Mb of memory is used for storing augmented music for `{g}` genre.")


#=================================For tabular features=============================


def extract_features_for_ml(track_arr,sr,g=None):
    """
        For ML feature extraction.
        Take a noisy music track and extract tabular features.
    """
    tempo = lf.tempo(y=track_arr,sr=sr).item()
    harm, perc = lb.effects.hpss(track_arr)
    harmony_mean = np.mean(harm) 
    harmony_var = np.var(harm)

    perceptr_mean = np.mean(perc) 
    perceptr_var = np.var(perc)
    
    rms = lf.rms(y=track_arr)
    rms_mean = np.mean(rms)
    rms_var = np.var(rms)
    
    chroma_stft = lf.chroma_stft(y=track_arr,sr=sr)
    chroma_stft_mean = np.mean(chroma_stft)
    chroma_stft_var = np.var(chroma_stft)
    
    spectral_centroid = lf.spectral_centroid(y=track_arr,sr=sr)
    spectral_centroid_mean = np.mean(spectral_centroid)
    spectral_centroid_var = np.var(spectral_centroid)


    spectral_bandwidth = lf.spectral_bandwidth(y=track_arr,sr=sr)
    spectral_bandwidth_mean = np.mean(spectral_bandwidth)
    spectral_bandwidth_var = np.var(spectral_bandwidth)

    spectral_rolloff = lf.spectral_rolloff(y=track_arr,sr=sr)
    spectral_rolloff_mean = np.mean(spectral_rolloff)
    spectral_rolloff_var = np.var(spectral_rolloff)

    zcr = lf.zero_crossing_rate(y=track_arr)
    zcr_mean = np.mean(zcr)
    zcr_var = np.var(zcr)

    mfccs = lf.mfcc(y=track_arr, sr=sr)
    mfcc_means = np.mean(mfccs,axis=1)
    mfcc_vars = np.var(mfccs,axis=1)

    features = {
        "tempo" : tempo,
        "harmony_mean" : harmony_mean,
        "harmony_var" : harmony_var,
        "perceptr_mean" : perceptr_mean,
        "perceptr_var" : perceptr_var,
        "rms_mean" : rms_mean,
        "rms_var" : rms_var,
        "chroma_stft_mean" : chroma_stft_mean,
        "chroma_stft_var" : chroma_stft_var,
        "spectral_centroid_mean" : spectral_centroid_mean,
        "spectral_centroid_var" : spectral_centroid_var,
        "spectral_bandwidth_mean" : spectral_bandwidth_mean,
        "spectral_bandwidth_var" : spectral_bandwidth_var,
        "spectral_rolloff_mean" : spectral_rolloff_mean,
        "spectral_rolloff_var" : spectral_rolloff_var,
        "zcr_mean" : zcr_mean,
        "zcr_var" : zcr_var
        
    }
    for i in range(0,20):
        features[f'mfcc{i+1}_mean'] = mfcc_means[i]
        features[f'mfcc{i+1}_var'] = mfcc_vars[i]
    if g:
        features["label"] = g
    return features

def create_features_from_dataset(features_name):
    global SR
    ml_df = pd.DataFrame(
        index=range(1000),
        columns= features_name
    )
    genres = ['blues', 'classical', 'country', 'disco', 'hiphop','jazz', 'metal', 'pop', 'reggae', 'rock']
    data_paths = [
        '/kaggle/input/datasets/akashkumbhakar/blues/',
        '/kaggle/input/datasets/akashkumbhakar/classical/',
        '/kaggle/input/datasets/akashkumbhakar/country/',
        '/kaggle/input/datasets/akashkumbhakar/disco/',
        '/kaggle/input/datasets/akashkumbhakar/hiphop/',
        '/kaggle/input/datasets/akashkumbhakar/jazz-genre/',
        '/kaggle/input/datasets/akashkumbhakar/metal/',
        '/kaggle/input/datasets/akashkumbhakar/pop-genre/',
        '/kaggle/input/datasets/akashkumbhakar/reggae-genre/',
        '/kaggle/input/datasets/akashkumbhakar/rock-genre/'
    ]
    j = 0
    for data_path,g in zip(data_paths,genres):
        print(data_path,g)
        for i in tqdm(range(0,100),desc=f"Extraction for `{g}` genre ..."):
            path = data_path + f"mashup_{i}.wav"
            y,sr = lb.load(path,sr=SR)
            f = extract_features_for_ml(y,sr,g=g)
            ml_df.loc[j] = list(f.values())
            j = j + 1

    return ml_df

def create_features_from_testdataset(test_data_path,test_df,features_name):
    global SR
    test_features_df = pd.DataFrame(
        index = range(test_df.shape[0]),
        columns = features_name
    )
    k = 0
    for idx,file in tqdm(zip(test_df['id'],test_df['filename']), desc="Creating test data ...",total=len(test_df)):
        path = test_data_path + file
        y,sr = lb.load(path,sr=SR)
        features = extract_features_for_ml(y,sr)
        values = [idx] + list(features.values())
        test_features_df.loc[k] = values
        k = k + 1
        
    return test_features_df

print("✅ Defined successfully.")

✅ Defined successfully.


# Music augmentation

In [3]:
sl = build_dataset(DATA_ROOT)

Loading stems into dict..: 100%|██████████| 10/10 [10:41<00:00, 64.20s/it]

✅ Total number of stems music file loaded successfully :  4000


In [4]:
noise = load_noise_audios(root_dir,audio_folder,csv_file)

Loading Noise audio ... : 100%|██████████| 2000/2000 [01:12<00:00, 27.71it/s]


In [5]:
# TESTING 
mashup = stem_recombination(sl,'classical')
mash = add_noise(mashup,list(noise.values()))
Audio(mash,rate=SR)

In [11]:
GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop','jazz', 'metal', 'pop', 'reggae', 'rock'] 
g = GENRES[9]
data_augmentation(sl,list(noise.values()),g=g,sample_count=15000)

# Storing to kaggle hub
handle = f'akashkumbhakar/{g}-15000'
local_dataset= f'/kaggle/working/{g}'

# Create a new dataset
kagglehub.dataset_upload(handle, local_dataset)
import shutil
shutil.rmtree(local_dataset)

rock
Directory created at: /kaggle/working/rock


Creating noisy mashup: 100%|██████████| 15000/15000 [04:18<00:00, 58.11it/s]

🤺 18.48318614065647 Mb of memory is used for storing augmented music for `rock` genre.
Uploading Dataset https://api.kaggle.com/datasets/akashkumbhakar/rock-15000 ...


More than 50 files detected, creating a zip archive...
Starting upload for file /tmp/tmp5nlox1zp/archive.zip


Uploading: 100%|██████████| 19.8G/19.8G [07:12<00:00, 45.9MB/s]  

Upload successful: /tmp/tmp5nlox1zp/archive.zip (18GB)


Your dataset has been created.
Files are being processed...
See at: https://api.kaggle.com/datasets/akashkumbhakar/rock-15000
